In [1]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"D:\project\DS\Job Market & Salary Analysis project"
)

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Always start cleaning from the original raw dataset
df = pd.read_csv(RAW_DIR / "postings.csv")

print("Initial dataset shape:", df.shape)
print("Duplicate job IDs:", df["job_id"].duplicated().sum())

print("\nMissing values in important columns:")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing_Count")
    .head(15)
)


Initial dataset shape: (123849, 31)
Duplicate job IDs: 0

Missing values in important columns:


,Missing_Count
closed_time,122776
skills_desc,121410
med_salary,117569
remote_allowed,108603
applies,100529
min_salary,94056
max_salary,94056
currency,87776
compensation_type,87776
pay_period,87776


In [2]:

# Missing-value report
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (
        df.isna().sum() / len(df) * 100
    ).round(2),
    "Data_Type": df.dtypes.astype(str)
})

missing_report = (
    missing_report[missing_report["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

display(missing_report)


,Missing_Count,Missing_Percentage,Data_Type
closed_time,122776,99.13,float64
skills_desc,121410,98.03,str
med_salary,117569,94.93,float64
remote_allowed,108603,87.69,float64
applies,100529,81.17,float64
min_salary,94056,75.94,float64
max_salary,94056,75.94,float64
pay_period,87776,70.87,str
compensation_type,87776,70.87,str
normalized_salary,87776,70.87,float64


In [3]:

# Compare salary field availability
salary_columns = [
    "min_salary",
    "max_salary",
    "med_salary",
    "normalized_salary"
]

salary_availability = pd.DataFrame({
    "Available_Count": df[salary_columns].notna().sum(),
    "Missing_Count": df[salary_columns].isna().sum()
})

display(salary_availability)

# Count postings with at least one salary value
has_any_salary = df[salary_columns].notna().any(axis=1)

print("Postings with any salary:", has_any_salary.sum())
print("Postings with no salary fields:", (~has_any_salary).sum())


,Available_Count,Missing_Count
min_salary,29793,94056
max_salary,29793,94056
med_salary,6280,117569
normalized_salary,36073,87776


Postings with any salary: 36073
Postings with no salary fields: 87776


In [4]:

# Work on a separate copy of the raw data
clean_df = df.copy()

# Identify text columns
text_columns = clean_df.select_dtypes(
    include=["object", "str", "string"]
).columns

# Remove leading/trailing whitespace and normalize empty strings
for col in text_columns:
    clean_df[col] = clean_df[col].str.strip()
    clean_df[col] = clean_df[col].replace("", pd.NA)

print("Text columns standardized:", len(text_columns))

# Check for empty strings remaining
empty_strings = (
    clean_df[text_columns]
    .eq("")
    .sum()
    .sum()
)

print("Empty strings remaining:", empty_strings)


Text columns standardized: 15
Empty strings remaining: 0


In [5]:

# Recheck missing values after standardization
display(
    clean_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)


closed_time                   122776
skills_desc                   121410
med_salary                    117569
remote_allowed                108603
applies                       100529
min_salary                     94056
max_salary                     94056
currency                       87776
compensation_type              87776
pay_period                     87776
normalized_salary              87776
posting_domain                 39968
application_url                36665
formatted_experience_level     29409
fips                           27415
dtype: int64

Original shape: (123849, 31)
Cleaned shape: (123849, 31)


In [6]:

# 1. Inspect data types
display(
    clean_df.dtypes.astype(str).to_frame("Data_Type")
)

# 2. Check salary range consistency
salary_range_invalid = (
    clean_df["min_salary"].notna()
    & clean_df["max_salary"].notna()
    & (clean_df["min_salary"] > clean_df["max_salary"])
)

print(
    "Minimum salary greater than maximum:",
    salary_range_invalid.sum()
)

# 3. Check negative values in salary columns
salary_columns = [
    "min_salary",
    "max_salary",
    "med_salary",
    "normalized_salary"
]

for col in salary_columns:
    negative_count = (clean_df[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

# 4. Check pay-period and currency categories
print("\nPay-period categories:")
display(clean_df["pay_period"].value_counts(dropna=False))

print("\nCurrency categories:")
display(clean_df["currency"].value_counts(dropna=False))


,Data_Type
job_id,int64
company_name,str
title,str
description,str
max_salary,float64
pay_period,str
location,str
company_id,float64
views,float64
med_salary,float64


Minimum salary greater than maximum: 0
min_salary: 0 negative values
max_salary: 0 negative values
med_salary: 0 negative values
normalized_salary: 0 negative values

Pay-period categories:


pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64


Currency categories:


currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64

In [7]:

# Check whether minimum salary exceeds maximum salary
invalid_range = (
    clean_df["min_salary"].notna()
    & clean_df["max_salary"].notna()
    & (clean_df["min_salary"] > clean_df["max_salary"])
)

print("Invalid salary ranges:", invalid_range.sum())

# Check for negative salary values
salary_columns = [
    "min_salary",
    "max_salary",
    "med_salary",
    "normalized_salary"
]

for col in salary_columns:
    print(f"{col}: {(clean_df[col] < 0).sum()} negative values")

# Inspect pay-period categories
print("\nPay-period distribution:")
display(clean_df["pay_period"].value_counts(dropna=False))

# Inspect currencies
print("\nCurrency distribution:")
display(clean_df["currency"].value_counts(dropna=False))


Invalid salary ranges: 0
min_salary: 0 negative values
max_salary: 0 negative values
med_salary: 0 negative values
normalized_salary: 0 negative values

Pay-period distribution:


pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64


Currency distribution:


currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64

In [8]:

# Count salary records for each pay period
salary_summary = (
    clean_df.groupby("pay_period", dropna=False)
    .agg(
        total_postings=("job_id", "count"),
        min_salary_available=("min_salary", "count"),
        max_salary_available=("max_salary", "count"),
        median_salary_available=("med_salary", "count"),
        normalized_salary_available=("normalized_salary", "count")
    )
)

display(salary_summary)


,total_postings,min_salary_available,max_salary_available,median_salary_available,normalized_salary_available
pay_period,,,,,
BIWEEKLY,9,9,9,0,9
HOURLY,14741,10212,10212,4529,14741
MONTHLY,518,288,288,230,518
WEEKLY,177,177,177,0,177
YEARLY,20628,19107,19107,1521,20628
NaN,87776,0,0,0,0


In [9]:

# Inspect salary ranges by pay period
salary_range_summary = (
    clean_df.groupby("pay_period")
    .agg(
        min_salary_value=("min_salary", "min"),
        median_of_min_salary=("min_salary", "median"),
        median_of_max_salary=("max_salary", "median"),
        max_salary_value=("max_salary", "max"),
        max_normalized_salary=("normalized_salary", "max")
    )
)

display(salary_range_summary)


,min_salary_value,median_of_min_salary,median_of_max_salary,max_salary_value,max_normalized_salary
pay_period,,,,,
BIWEEKLY,2057.0,54009.0,66976.0,89965.0,2096744.0
HOURLY,7.0,25.0,32.0,275000.0,535600000.0
MONTHLY,1.0,5550.0,7350.0,110000.0,1200000.0
WEEKLY,977.0,2099.0,2099.0,4049.0,210548.0
YEARLY,1.0,87000.0,120000.0,120000000.0,102500000.0


In [10]:

# Inspect the 10 largest normalized salary values
salary_columns = [
    "job_id",
    "title",
    "company_name",
    "pay_period",
    "currency",
    "min_salary",
    "med_salary",
    "max_salary",
    "normalized_salary"
]

suspicious_salary = (
    clean_df[
        clean_df["normalized_salary"].notna()
    ]
    .nlargest(10, "normalized_salary")[salary_columns]
)

display(suspicious_salary)


,job_id,title,company_name,pay_period,currency,min_salary,med_salary,max_salary,normalized_salary
9237,3886471794,Intellectual Property Associate (246215),Eastridge Workforce Solutions,HOURLY,USD,240000.0,NaN,275000.0,535600000.0
98888,3904989090,"Case Manager RN, Pedi Rheumatology",Kaiser Permanente,HOURLY,USD,151900.0,NaN,196570.0,362408800.0
13614,3887837646,Cloud Domain Architect,Applicantz,HOURLY,USD,125000.0,NaN,150000.0,286000000.0
89082,3904407236,Cloud Domain Architect,Applicantz,HOURLY,USD,125000.0,NaN,150000.0,286000000.0
60038,3901967159,Billing Supervisor (246521),Eastridge Workforce Solutions,HOURLY,USD,125000.0,NaN,125000.0,260000000.0
102481,3905282337,Manager - Clinical Applications Support - Anci...,Lakeland Regional Health-Florida,HOURLY,USD,98467.0,NaN,123094.0,230423440.0
71845,3902865442,Sharepoint Developer,"DigiDoc, Inc. dba Public Sector Solutions Group",HOURLY,USD,100000.0,NaN,120000.0,228800000.0
72585,3902879426,Trial Paralegal - Hybrid in OC or LA!,Adams & Martin Group,HOURLY,USD,80000.0,NaN,100000.0,187200000.0
17646,3888406456,DCS Technician,Jobot,HOURLY,USD,70000.0,NaN,80000.0,156000000.0
87241,3904388630,Full Time Field Service Technician – Anaheim/P...,The Hillman Group,HOURLY,USD,62000.0,NaN,68000.0,135200000.0


In [11]:

# Investigate the relationship between raw and normalized salary

salary_check = clean_df.loc[
    clean_df["normalized_salary"].notna()
    & clean_df["min_salary"].notna()
    & clean_df["max_salary"].notna(),
    [
        "job_id",
        "title",
        "pay_period",
        "min_salary",
        "max_salary",
        "normalized_salary"
    ]
].copy()

salary_check["salary_midpoint"] = (
    salary_check["min_salary"] + salary_check["max_salary"]
) / 2

salary_check["normalization_ratio"] = (
    salary_check["normalized_salary"]
    / salary_check["salary_midpoint"].replace(0, float("nan"))
)

display(
    salary_check
    .sort_values("normalized_salary", ascending=False)
    .head(10)
)



,job_id,title,pay_period,min_salary,max_salary,normalized_salary,salary_midpoint,normalization_ratio
9237,3886471794,Intellectual Property Associate (246215),HOURLY,240000.0,275000.0,535600000.0,257500.0,2080.0
98888,3904989090,"Case Manager RN, Pedi Rheumatology",HOURLY,151900.0,196570.0,362408800.0,174235.0,2080.0
13614,3887837646,Cloud Domain Architect,HOURLY,125000.0,150000.0,286000000.0,137500.0,2080.0
89082,3904407236,Cloud Domain Architect,HOURLY,125000.0,150000.0,286000000.0,137500.0,2080.0
60038,3901967159,Billing Supervisor (246521),HOURLY,125000.0,125000.0,260000000.0,125000.0,2080.0
102481,3905282337,Manager - Clinical Applications Support - Anci...,HOURLY,98467.0,123094.0,230423440.0,110780.5,2080.0
71845,3902865442,Sharepoint Developer,HOURLY,100000.0,120000.0,228800000.0,110000.0,2080.0
72585,3902879426,Trial Paralegal - Hybrid in OC or LA!,HOURLY,80000.0,100000.0,187200000.0,90000.0,2080.0
17646,3888406456,DCS Technician,HOURLY,70000.0,80000.0,156000000.0,75000.0,2080.0
87241,3904388630,Full Time Field Service Technician – Anaheim/P...,HOURLY,62000.0,68000.0,135200000.0,65000.0,2080.0


In [12]:

# Check hourly postings with the suspicious 2,080 normalization ratio

hourly_check = salary_check[
    salary_check["pay_period"] == "HOURLY"
].copy()

hourly_check["ratio_is_2080"] = (
    hourly_check["normalization_ratio"].round(2) == 2080
)

print("Hourly records with comparable salary fields:",
      len(hourly_check))

print("Records with ratio 2080:",
      hourly_check["ratio_is_2080"].sum())

print("Percentage with ratio 2080:",
      round(hourly_check["ratio_is_2080"].mean() * 100, 2), "%")


Hourly records with comparable salary fields: 10212
Records with ratio 2080: 10212
Percentage with ratio 2080: 100.0 %


In [13]:

# Flag hourly salary records with a suspicious normalization ratio

clean_df["salary_quality_flag"] = "Review not required"

hourly_mask = (
    clean_df["pay_period"].eq("HOURLY")
    & clean_df["min_salary"].notna()
    & clean_df["max_salary"].notna()
    & clean_df["normalized_salary"].notna()
)

hourly_salary_midpoint = (
    clean_df["min_salary"] + clean_df["max_salary"]
) / 2

ratio_2080_mask = (
    hourly_mask
    & (
        (
            clean_df["normalized_salary"]
            / hourly_salary_midpoint.replace(0, float("nan"))
        ).round(2) == 2080
    )
)

clean_df.loc[
    ratio_2080_mask,
    "salary_quality_flag"
] = "Review: hourly normalization ratio 2080"

print("Records flagged:", ratio_2080_mask.sum())

print("\nFlag distribution:")
display(clean_df["salary_quality_flag"].value_counts())



Records flagged: 10212

Flag distribution:


salary_quality_flag
Review not required                        113637
Review: hourly normalization ratio 2080     10212
Name: count, dtype: int64

In [14]:

# Review data types before finalizing the cleaned dataset

print("Dataset shape:", clean_df.shape)

print("\nColumn data types:")
display(clean_df.dtypes.to_frame(name="data_type"))

print("\nIdentifier samples:")
for col in ["company_id", "zip_code", "fips"]:
    print(f"\n{col}:")
    display(clean_df[col].dropna().head(5))


Dataset shape: (123849, 32)

Column data types:


,data_type
job_id,int64
company_name,str
title,str
description,str
max_salary,float64
pay_period,str
location,str
company_id,float64
views,float64
med_salary,float64



Identifier samples:

company_id:


0     2774458.0
2    64896719.0
3      766262.0
5     1481176.0
6    81942316.0
Name: company_id, dtype: float64


zip_code:


0     8540.0
1    80521.0
2    45202.0
3    11040.0
4    52601.0
Name: zip_code, dtype: float64


fips:


0    34021.0
1     8069.0
2    39061.0
3    36059.0
4    19057.0
Name: fips, dtype: float64

In [15]:

# Convert identifier columns to strings while preserving missing values

identifier_columns = ["company_id", "zip_code", "fips"]

for col in identifier_columns:
    clean_df[col] = (
        clean_df[col]
        .map(lambda x: str(int(x)) if pd.notna(x) else pd.NA)
        .astype("string")
    )

# Verify the changes
print("Updated data types:")
display(clean_df[identifier_columns].dtypes)

print("\nSample identifier values:")
display(clean_df[identifier_columns].head(10))

print("\nDataset shape:", clean_df.shape)


Updated data types:


company_id    string
zip_code      string
fips          string
dtype: object


Sample identifier values:


,company_id,zip_code,fips
0,2774458,8540,34021
1,<NA>,80521,8069
2,64896719,45202,39061
3,766262,11040,36059
4,<NA>,52601,19057
5,1481176,27601,37183
6,81942316,<NA>,<NA>
7,<NA>,94101,6075
8,721189,68102,31055
9,28631247,32905,12009



Dataset shape: (123849, 32)


In [16]:

# 1. Check dataset dimensions
print("Dataset shape:", clean_df.shape)

# 2. Check duplicate job IDs
print("Duplicate job IDs:", clean_df["job_id"].duplicated().sum())

# 3. Check empty strings in text columns
text_columns = clean_df.select_dtypes(
    include=["object", "str", "string"]
).columns

empty_strings = (
    clean_df[text_columns]
    .eq("")
    .sum()
    .sum()
)

print("Empty strings remaining:", empty_strings)

# 4. Check salary-quality flags
print("\nSalary-quality flag counts:")
display(clean_df["salary_quality_flag"].value_counts(dropna=False))

# 5. Review missing values in the most incomplete columns
print("\nMissing-value counts:")
display(
    clean_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


Dataset shape: (123849, 32)
Duplicate job IDs: 0
Empty strings remaining: 0

Salary-quality flag counts:


salary_quality_flag
Review not required                        113637
Review: hourly normalization ratio 2080     10212
Name: count, dtype: int64


Missing-value counts:


closed_time          122776
skills_desc          121410
med_salary           117569
remote_allowed       108603
applies              100529
max_salary            94056
min_salary            94056
pay_period            87776
compensation_type     87776
currency              87776
dtype: int64

In [17]:

from pathlib import Path

# Project's processed-data directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
output_path = processed_dir / "cleaned_job_postings.csv"

clean_df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File path:", output_path.resolve())
print("Rows saved:", len(clean_df))
print("Columns saved:", len(clean_df.columns))
print("File exists:", output_path.exists())


Cleaned dataset saved successfully!
File path: D:\project\DS\Job Market & Salary Analysis project\DATA\Processed\cleaned_job_postings.csv
Rows saved: 123849
Columns saved: 32
File exists: True


In [18]:

# Reload the saved CSV and verify it
saved_df = pd.read_csv(output_path)

print("Saved dataset shape:", saved_df.shape)
print("Duplicate job IDs:", saved_df["job_id"].duplicated().sum())
print("Salary flag column exists:",
      "salary_quality_flag" in saved_df.columns)


Saved dataset shape: (123849, 32)
Duplicate job IDs: 0
Salary flag column exists: True
